In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
train_dir = '/kaggle/input/datasets/tongpython/cat-and-dog/training_set/training_set'
test_dir = '/kaggle/input/datasets/tongpython/cat-and-dog/test_set/test_set'

In [13]:
from torch.utils.data import Subset

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_full = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_full = datasets.ImageFolder(root=train_dir, transform=test_transforms)

g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_full), generator=g)
n_val = int(0.1 * len(train_full))
val_idx = indices[:n_val]
train_idx = indices[n_val:]

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Train: 7205, Val: 800, Test: 2023


In [14]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 3, 224, 224])
Labels shape: torch.Size([64])


In [15]:
def nin_block(in_channels, out_channels, kernel_size, strides, padding):
      return nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding, bias=False),
          nn.BatchNorm2d(out_channels),
          nn.ReLU(inplace=True),
          nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
          nn.BatchNorm2d(out_channels),
          nn.ReLU(inplace=True),
          nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
          nn.BatchNorm2d(out_channels),
          nn.ReLU(inplace=True),
      )

In [16]:
net = nn.Sequential(
      nin_block(3, 96, kernel_size=5, strides=1, padding=2),
      nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
      nin_block(96, 192, kernel_size=3, strides=1, padding=1),
      nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
      nin_block(192, 384, kernel_size=3, strides=1, padding=1),
      nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
      nn.Dropout(0.5),
      nn.Conv2d(384, 2, kernel_size=1),
      nn.AdaptiveAvgPool2d((1, 1)),
      nn.Flatten()
  )

In [17]:
import torch.optim as optim

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

net = net.to(device)
criterion = nn.CrossEntropyLoss()
lr = 1e-3
num_epochs = 20
optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
print("Device:", device)

Device: cuda


In [18]:
from torch.amp import autocast, GradScaler

scaler = GradScaler('cuda')

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=10, start_epoch=0, best_acc=0.0):
    for epoch in range(start_epoch, start_epoch + num_epochs):
        # --- train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        # --- val ---
        model.eval()
        v_correct, v_total, v_loss = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                v_loss += loss.item() * images.size(0)
                v_correct += (outputs.argmax(dim=1) == labels).sum().item()
                v_total += labels.size(0)
        val_loss, val_acc = v_loss / v_total, v_correct / v_total
        scheduler.step(val_acc)

        print(
            f"Epoch [{epoch + 1}/{start_epoch + num_epochs}] "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )


        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_acc": max(best_acc, val_acc),
        }, "last_nin.pth")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_nin.pth")

    print(f"Best Val Acc: {best_acc:.4f}")
    model.load_state_dict(torch.load("best_nin.pth"))
    return model


def load_checkpoint(model, optimizer, scheduler, path="last_nin.pth"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    print(f"Resume từ epoch {ckpt['epoch']}, best_acc={ckpt['best_acc']:.4f}")
    return ckpt["epoch"], ckpt["best_acc"]

In [19]:
net = train_model(
    model=net,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=num_epochs
)

Epoch [1/20] Train Loss: 0.6555 Acc: 0.6114 | Val Loss: 0.6525 Acc: 0.6075 LR: 0.001000
Epoch [2/20] Train Loss: 0.6161 Acc: 0.6545 | Val Loss: 0.6222 Acc: 0.6787 LR: 0.001000
Epoch [3/20] Train Loss: 0.5861 Acc: 0.6890 | Val Loss: 0.5879 Acc: 0.6937 LR: 0.001000
Epoch [4/20] Train Loss: 0.5694 Acc: 0.7017 | Val Loss: 0.5846 Acc: 0.6963 LR: 0.001000
Epoch [5/20] Train Loss: 0.5511 Acc: 0.7189 | Val Loss: 0.5917 Acc: 0.6975 LR: 0.001000
Epoch [6/20] Train Loss: 0.5388 Acc: 0.7277 | Val Loss: 0.6304 Acc: 0.6763 LR: 0.001000
Epoch [7/20] Train Loss: 0.5262 Acc: 0.7375 | Val Loss: 0.5261 Acc: 0.7475 LR: 0.001000
Epoch [8/20] Train Loss: 0.5075 Acc: 0.7535 | Val Loss: 0.5304 Acc: 0.7475 LR: 0.001000
Epoch [9/20] Train Loss: 0.4982 Acc: 0.7598 | Val Loss: 0.5342 Acc: 0.7538 LR: 0.001000
Epoch [10/20] Train Loss: 0.4952 Acc: 0.7674 | Val Loss: 0.5013 Acc: 0.7762 LR: 0.001000
Epoch [11/20] Train Loss: 0.4798 Acc: 0.7717 | Val Loss: 0.7093 Acc: 0.6800 LR: 0.001000
Epoch [12/20] Train Loss: 0.47

In [20]:
def evaluate(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    return accuracy

test_acc = evaluate(net, test_loader, device)

Test Accuracy: 0.8196
